<div dir="rtl" style="text-align:right">
 پروژهٔ تحلیل تصادف از ویدئو (دانلود → ایندکس → استخراج فریم → آموزش → تست)

**هدف:** بخش اول:
1) دانلود دیتاست از هاگینگ‌فیس (در صورت نیاز)  
2) ساخت فایل ایندکس با استریم (بدون دیکود ویدئو)  
3) استخراج فریم‌ها با `ffmpeg`  
4) ساخت دیتاست/لودر  
5) آموزش بیس‌لاین سبک با **ResNet18 (یک فریم)**  
6) ارزیابی و ذخیرهٔ بهترین مدل در `best.pth`  
7) تست روی یک **ویدئوی محلی**
</div>

<div dir="rtl" style="text-align:right">
نصب پیش‌نیازها (در صورت نیاز)
</div>

In [1]:

#!pip -q install huggingface_hub 
#!pip -q install datasets 
#!pip -q install tqdm
#!apt-get -y install ffmpeg


<div dir="rtl" style="text-align:right">
 ایمپورت‌ها و تنظیمات پایه
</div>

In [ ]:

import os, csv, subprocess
from pathlib import Path
from typing import Optional

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

from datasets import load_dataset, Value
from huggingface_hub import snapshot_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("nexar_full")
INDEX_CSV = "nexar_index_colab.csv"
FRAMES_DIR = Path("frames_phase1")
FRAMES_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-4
SEED = 42

torch.manual_seed(SEED)


' import os, csv, subprocess\nfrom pathlib import Path\nfrom typing import Optional\n\nimport pandas as pd\nfrom PIL import Image\nfrom tqdm.auto import tqdm\n\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom torch.utils.data import Dataset, DataLoader, random_split\nfrom torchvision import transforms, models\n\nfrom datasets import load_dataset, Value\nfrom huggingface_hub import snapshot_download\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\nDATA_ROOT = Path("nexar_full")\nINDEX_CSV = "nexar_index_colab.csv"\nFRAMES_DIR = Path("frames_phase1")\nFRAMES_DIR.mkdir(exist_ok=True)\n\nBATCH_SIZE = 8\nEPOCHS = 5\nLR = 1e-4\nSEED = 42\n\ntorch.manual_seed(SEED) '

<div dir="rtl" style="text-align:right">
 ۱) دانلود دیتاست (در صورت نیاز)
اگر قبلاً دانلود کرده‌اید، این مرحله را نادیده بگیرید. 
</div>

In [ ]:

REPO_ID = "nexar-ai/nexar_collision_prediction"
if not DATA_ROOT.exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    local_dir = snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        local_dir=str(DATA_ROOT),
        local_dir_use_symlinks=False,
        allow_patterns=["train/**", "metadata.csv", "README.md"],
        token=os.environ.get("HF_TOKEN", None)
    )
    print("Downloaded to:", local_dir)
else:
    print("Dataset directory already exists:", DATA_ROOT)


<div dir="rtl" style="text-align:right">
 ۲) ساخت فایل ایندکس با استریم
ستون `video` به رشته تبدیل می‌شود و `label` بر اساس وجود `time_of_event` تعیین می‌شود.
</div>

In [ ]:

if not Path(INDEX_CSV).exists():
    ds = load_dataset(REPO_ID, split="train", streaming=True)
    ds = ds.cast_column("video", Value("string"))
    import csv
    with open(INDEX_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["video_path","label","time_of_event","time_of_alert","light_conditions","weather","scene","time_to_accident"])
        cnt = 0
        for item in ds:
            lbl = 1 if item.get("time_of_event") is not None else 0
            w.writerow([
                item.get("video"),
                lbl,
                item.get("time_of_event"),
                item.get("time_of_alert"),
                item.get("light_conditions"),
                item.get("weather"),
                item.get("scene"),
                item.get("time_to_accident")
            ])
            cnt += 1
        print("Wrote index, count:", cnt)
else:
    print("Index already exists:", INDEX_CSV)

df = pd.read_csv(INDEX_CSV)
print("Index rows:", len(df))
print(df.head(5).to_string(index=False))


<div dir="rtl" style="text-align:right">
 ۳) استخراج فریم‌ها با FFmpeg
پنجرهٔ ۲۴ فریمی حول `time_of_event` برای مثبت‌ها و ابتدای ویدئو برای منفی‌ها.
</div>

In [ ]:

def hf_url_to_local(hf_url: str) -> Path:
    rel = hf_url.split('@', 1)[-1].split('/', 1)[-1]
    return DATA_ROOT / rel

def video_stem(hf_url: str) -> str:
    return os.path.basename(hf_url).replace(".mp4","")

def ffprobe_duration(video_path: Path) -> Optional[float]:
    cmd = ["ffprobe","-v","error","-select_streams","v:0",
           "-show_entries","format=duration","-of","default=nw=1:nk=1", str(video_path)]
    try:
        out = subprocess.check_output(cmd).decode().strip()
        return float(out)
    except Exception:
        return None

def pick_window(time_of_event: Optional[float], duration: float, T: int = 24, fps: int = 6):
    if time_of_event is None:
        return 0.0, fps
    start = max(0.0, float(time_of_event) - (T/fps)/2)
    end = min(duration, start + (T/fps))
    start = max(0.0, end - (T/fps))
    return start, fps

def extract_frames_one(hf_url: str, time_of_event_val) -> bool:
    local_video = hf_url_to_local(hf_url)
    if not local_video.exists():
        print("Missing:", local_video)
        return False
    stem = video_stem(hf_url)
    out_dir = FRAMES_DIR / stem
    if out_dir.exists() and len(list(out_dir.glob("*.jpg"))) >= 24:
        return True
    out_dir.mkdir(parents=True, exist_ok=True)
    dur = ffprobe_duration(local_video)
    if not dur: return False
    start, fps = pick_window(time_of_event_val, dur, T=24, fps=6)
    cmd = ["ffmpeg","-ss", f"{start:.3f}","-t", f"{24/fps:.3f}","-i", str(local_video),
           "-vf", f"fps={fps},scale=224:224", str(out_dir/"frame_%05d.jpg"), "-y","-loglevel","error"]
    try:
        subprocess.run(cmd, check=True)
        return True
    except subprocess.CalledProcessError:
        return False

ok, fail = 0, 0
for _, row in tqdm(df.iterrows(), total=len(df)):
    path = row["video_path"]
    if pd.isna(path): continue
    ev = None if pd.isna(row["time_of_event"]) else row["time_of_event"]
    if extract_frames_one(path, ev): ok += 1
    else: fail += 1
print("Done. ok:", ok, "fail:", fail)


<div dir="rtl" style="text-align:right">
 ۴) دیتاست و لودر
تقسیم ۸۰/۲۰ و نرمال‌سازی استاندارد.
</div>

In [ ]:

class CollisionFramesDataset(torch.utils.data.Dataset):
    def __init__(self, index_csv, frames_root, transform=None, max_frames=24):
        self.items = []
        self.frames_root = Path(frames_root)
        self.transform = transform
        self.max_frames = max_frames
        import csv
        with open(index_csv, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                stem = os.path.basename(row["video_path"]).replace(".mp4","")
                label = int(row["label"])
                if (self.frames_root / stem).exists():
                    self.items.append((stem, label))
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        stem, label = self.items[idx]
        fps = sorted((self.frames_root / stem).glob("*.jpg"))[:self.max_frames]
        imgs = []
        for p in fps:
            img = Image.open(p).convert("RGB")
            if self.transform: img = self.transform(img)
            imgs.append(img)
        x = torch.stack(imgs)
        return x, label

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

full_ds = CollisionFramesDataset(INDEX_CSV, FRAMES_DIR, transform=transform, max_frames=24)
n_total = len(full_ds)
n_train = int(0.8*n_total)
train_ds, val_ds = random_split(full_ds, [n_train, n_total-n_train])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_loader))
print("Batch shape:", xb.shape, "Labels:", yb[:8])


<div dir="rtl" style="text-align:right">
 ۵) مدل ResNet18 تک‌فریم
ساده و سریع؛ مناسب برای بیس‌لاینِ.
</div>

In [ ]:

class ResNetSingleFrame(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        self.backbone = m
    def forward(self, x):
        x = x[:,0]
        return self.backbone(x)

model = ResNetSingleFrame().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


<div dir="rtl" style="text-align:right">
 ۶) آموزش و ارزیابی (ذخیرهٔ `best.pth`)
</div>

In [ ]:

def run_epoch(loader, train=True):
    model.train(train)
    tot, ok, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train: optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            loss_sum += loss.item()*y.size(0)
            ok += (logits.argmax(1)==y).sum().item()
            tot += y.size(0)
    return loss_sum/tot, ok/tot

best = 1e9
for ep in range(20):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    print(f"Epoch {ep+1}/5 — train_loss: {tr_loss:.4f}, train_acc: {tr_acc:.4f} — val_loss: {va_loss:.4f}, val_acc: {va_acc:.4f}")
    if va_loss < best:
        best = va_loss
        torch.save(model.state_dict(), "best.pth")
        print("Saved best model")


<div dir="rtl" style="text-align:right">
## ۷) تست سریع + تست روی یک ویدئوی محلی
مسیر ویدئو را در متغیر زیر تغییر دهید.
</div>

In [ ]:

# quick preview
model.load_state_dict(torch.load("best.pth", map_location=DEVICE))
model.eval()
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        p = model(x).argmax(1).cpu()
        print("Preds:", p[:16].tolist(), "Labels:", y[:16].tolist())
        break

# local video inference
LOCAL_VIDEO_PATH = "sample_local.mp4"  # change this
TMP_DIR = Path("tmp_infer_frames"); TMP_DIR.mkdir(exist_ok=True)
for p in TMP_DIR.glob("*.jpg"): p.unlink()

cmd = ["ffmpeg","-y","-i", LOCAL_VIDEO_PATH, "-vf","fps=1,scale=224:224", str(TMP_DIR/"frame_%05d.jpg")]
try:
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
except subprocess.CalledProcessError:
    print("ffmpeg failed. Check LOCAL_VIDEO_PATH / ffmpeg install.")
    raise

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
frames = sorted(TMP_DIR.glob("*.jpg"))[:24]
imgs = [tf(Image.open(p).convert("RGB")) for p in frames]
x = torch.stack(imgs).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = model(x)
    prob = torch.softmax(logits, dim=1)[0]
    pred = logits.argmax(1).item()

print(f"Pred: {pred}  (0=non-collision, 1=collision)")
print(f"Probs: non-collision={prob[0]:.3f}, collision={prob[1]:.3f}")
